# 第2章 GPU 体系结构（上）：编程模型与 wavefront 执行

grid/workgroup/wavefront/lane 的工作划分，WGP/CU/SIMD 落点，EXEC 掩码与分支发散

**平台**：AMD Radeon RX 9070 XT (`gfx1201`) + ROCm 7.13 + Ubuntu 24.04

**代码目录**: `code/part0-intro/chapter2/`

## 在云端运行本章

本教程以 RX 9070 XT（`gfx1201` / RDNA4）为讲解和参考环境。其他 GPU 可以跟随本章理解线程、wavefront 和分支发散；请将性能观察限定在同一平台内，不与参考数据直接比较。

云平台已预装 ROCm、PyTorch 和基础编译工具，可直接使用；本地读者仍按下文完成环境准备。本章如果需要额外依赖，会在对应步骤单独提示。

编译前请根据当前 `rocminfo` 输出确认实际 `gfx` 架构；未识别时先检查环境。


## 本章导读

> 第 1 章我们确认了环境是通的——ROCm 看得到 GPU、PyTorch 用得上 GPU、最小 HIP 程序能编译运行，环境这张地图已经在你手上。现在该铺开第二张地图了：**GPU 体系结构**。本章会做三件事：跟着一次真实的 kernel 提交（`hipLaunchKernelGGL`）看清线程怎么被划分（grid → workgroup → wavefront → lane）、搞清楚这些 wavefront 落到哪块硬件上执行（WGP/CU/SIMD）、再用 EXEC 掩码弄懂分支发散为什么会让一整排线程被拖住。
>
> 这套「软件怎么划分、硬件怎么执行」的两层视角，是后面一切优化的心智地基：下一章会接着讲片上资源和内存层级；Part 2 的每个算子优化——合并访存（第 8 章 Element-Wise）、跨线程归约（第 9 章 Reduction）、分块与寄存器累加（第 11 章 GEMM）、矩阵指令与融合（第 12 章 Attention/Fusion）——都建立在它之上。本章只负责把模型立起来，具体怎么优化，留到对应算子章。

本章对应代码在：

```text
code/part0-intro/
├── pyproject.toml
├── uv.lock
├── activate-rocm.sh
└── chapter2/
    ├── branch_divergence.hip   # 选做：分支发散受控对照
    └── run_all.sh
```

## 硬件规格与术语对照

本章所有数字都锚定在一块具体的卡上：**AMD Radeon RX 9070 XT**。按 AMD 官方规格，它有 64 个计算单元（Compute Unit，CU）、16 GB 显存（GDDR6，256-bit 位宽）、最高约 640 GB/s 的显存带宽，以及 64 MB 的 Infinity Cache。

ROCm 文档里 `gfx1201` 这一条还给出了更细的资源数字：支持 wave32/wave64、128 KiB 片上内存（LDS）、768 KiB 向量寄存器和 32 KiB 标量寄存器。

> **请注意：这些是厂家标称的规格，不是我们实测跑出来的速度。**

*（图示：一次 HIP kernel 的全景——从 launch 到 EXEC 的前半段为本章内容，资源与内存见第 3 章）*

```mermaid
flowchart LR
    H[Host: hipLaunchKernelGGL] --> G[grid]
    G --> WG[workgroup / block]
    WG --> WF[wavefront: wave32 or wave64]
    WF --> PL[WGP placement]
    PL --> EX[CU-mode or WGP-mode execution]
    EX --> SIMD[SIMD lanes execute vector instructions]
    SIMD --> MASK[EXEC selects active lanes]
    SIMD --> RES[VGPR / SGPR / LDS constrain residency]
    RES --> MEM[registers, LDS, caches, GDDR6]
```

这张图是**从「你写的代码」到「硬件怎么执行」的导览图**，不是某一次运行的真实录像。具体会落到哪个硬件单元、同时运行几个 wave、缓存是否命中，都要结合编译产物和 profiling 工具确认，读图时请把它当作执行关系的概览。

### CUDA ↔ HIP ↔ AMD 执行模型术语对照

| HIP 里的词 | CUDA 里常见的词 | AMD 执行模型里的词 | 本章怎么理解它 |
| --- | --- | --- | --- |
| thread / work-item | thread | lane（通道）上的一份工作 | 只是一个逻辑编号，不等于一颗独立的处理器 |
| block | thread block | workgroup（工作组） | 一组能互相协作、能共用片上内存（LDS）的线程 |
| warp | warp | wavefront | 同一 wavefront 的线程步调一致，执行同一段指令；本机是 wave32，也可能是 wave64 |
| grid | grid | 一次提交（dispatch）里的全部工作组 | 你交给 GPU 的全部工作，但不保证它们立刻同时开跑 |
| `__shared__` | shared memory | LDS（局部数据存储，Local Data Share） | 工作组能自己支配的片上内存 |

> **读法约定**：下文说的「RX 9070 XT」，只指我们实测过的那台 `gfx1201` 机器；「wave32」既是这台机器在实验里测到的真实 wavefront 大小，也是本章代码用的分组。换用其他 AMD 显卡、编译选项或 wave64 kernel 时，请重新测量并单独解读。

## 环境准备

### 1. 定位仓库根目录

In [ ]:
# 检测当前 GPU 架构（仅用于 hipcc 编译 target）
import os
import re
import subprocess

ARCH_DETECT_TIMEOUT_S = 10
COMPILE_TIMEOUT_S = 120
SMOKE_TIMEOUT_S = 120
FULL_TIMEOUT_S = 300
SUPPORTED_ARCHES = {"gfx1100", "gfx1151", "gfx1201"}
GPU_AGENT_NAME_RE = re.compile(r"(?m)^\s*Name:\s*(gfx[0-9a-z]+)\s*$")
GPU_AGENT_BLOCK_RE = re.compile(
    r"(?ms)^\s*Agent\s+\d+\s*$.*?(?=^\s*Agent\s+\d+\s*$|\Z)"
)


def _gpu_agent_arches(rocminfo_stdout):
    """Return exact gfx names from GPU Agent blocks only."""
    candidates = []
    for block in GPU_AGENT_BLOCK_RE.findall(rocminfo_stdout):
        if re.search(r"(?m)^\s*Device Type:\s*GPU\s*$", block):
            candidates.extend(GPU_AGENT_NAME_RE.findall(block))
    return sorted(set(candidates))


def run_checked(command, *, cwd=None, timeout=SMOKE_TIMEOUT_S):
    """Run a command, expose failures, and stop before stale results are used."""
    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            check=False,
            cwd=cwd,
            timeout=timeout,
        )
    except subprocess.TimeoutExpired as exc:
        print(f"命令超时（timeout={timeout}s）: {' '.join(command)}")
        if exc.stdout:
            print(f"stdout:\n{exc.stdout}")
        if exc.stderr:
            print(f"stderr:\n{exc.stderr}")
        raise RuntimeError("外部命令超时，已停止后续步骤") from exc
    except FileNotFoundError as exc:
        print(f"命令不存在: {command[0]}")
        raise RuntimeError("外部命令不可用，已停止后续步骤") from exc
    if completed.returncode != 0:
        print(f"命令失败（returncode={completed.returncode}）: {' '.join(command)}")
        if completed.stdout:
            print(f"stdout:\n{completed.stdout}")
        if completed.stderr:
            print(f"stderr:\n{completed.stderr}")
        raise RuntimeError("外部命令失败，已停止后续步骤")
    return completed


override_arch = os.environ.get("HELLO_GPU_ARCH", "").strip()
if override_arch:
    if override_arch not in SUPPORTED_ARCHES:
        raise ValueError(
            f"HELLO_GPU_ARCH 仅支持 {sorted(SUPPORTED_ARCHES)}；实际值={override_arch!r}"
        )
    arch = override_arch
    arch_source = "override"
else:
    rocminfo_result = run_checked(["rocminfo"], timeout=ARCH_DETECT_TIMEOUT_S)
    gpu_arches = _gpu_agent_arches(rocminfo_result.stdout)
    if len(gpu_arches) != 1:
        raise RuntimeError(
            "rocminfo 必须恰好报告一个 GPU Agent Name: gfx...，"
            f"实际候选={gpu_arches or 'none'}"
        )
    arch = gpu_arches[0]
    arch_source = "rocminfo"

print(f"arch: {arch}")
print(f"arch_source: {arch_source}")


In [ ]:
import pathlib
import subprocess

def find_repo_root():
    current = pathlib.Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (
            (candidate / "README.md").exists()
            and (candidate / "code").is_dir()
            and (candidate / "docs").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise FileNotFoundError("无法定位仓库根目录")

REPO_ROOT = find_repo_root()
print(f"仓库根目录: {REPO_ROOT}")

### 2. 检测 GPU 架构

运行 `rocminfo` 检测当前 GPU 架构（精确解析 GPU Agent 的 `Name: gfx...`）；也可用 `HELLO_GPU_ARCH` 仅覆盖 hipcc 编译 target。上方代码会打印 `arch` 与 `arch_source`；如果没有识别到唯一架构，请先检查环境，或填写已经确认过的 `HELLO_GPU_ARCH` 后再继续。

## 2.1 从 launch 到 workgroup

**这一节只解决一件事：你写下的一大堆线程，是怎么被分成组的。**

我们在 CPU 这边启动 kernel 时，会明确告诉它两件事：一共要算多少组（grid）、每组多少个线程（block）。本仓库的全局内存实验把每组定为 `kBlockSize=256` 个线程，于是代码写成 `dim3 grid(grid_for(n))` 和 `dim3 block(kBlockSize)`，再一起交给 `hipLaunchKernelGGL`。kernel 里面，则用下面这行算出「我是第几号线程」：

```cpp
std::size_t tid = blockIdx.x * blockDim.x + threadIdx.x;
if (tid < n) output[tid] = input[tid];
```

这样，每个线程（work-item）就对上了一个全局编号。这里要特别提醒一句：`block` 只是**软件上的分组**，它只告诉运行时「每组多少个线程」，**并没有**承诺「一组 block 就对应一个硬件计算单元（CU）」，也没承诺这些组会按你提交的顺序完成。

把这套划分画成一张层级图，一眼就能看全：grid 切成 workgroup，workgroup 再切成 wavefront，每个 lane 对应一个全局下标。

*（图示：HIP 的线程层级——grid 切成 workgroup，workgroup 再切成 wavefront（本机为 wave32），每个 lane 对应一个全局下标）*

```mermaid
flowchart TB
    G["grid：一次提交的全部工作"] --> WG0["workgroup 0（blockIdx.x = 0）"]
    G --> WG1["workgroup 1（blockIdx.x = 1）"]
    G --> WGN["……更多 workgroup"]
    WG0 --> WF0["wavefront 0：lanes 0–31"]
    WG0 --> WFD["……"]
    WG0 --> WF7["wavefront 7：lanes 224–255"]
    WF0 --> L0["lane 0 → tid 0"]
    WF0 --> L1["lane 1 → tid 1"]
    WF0 --> LD["……"]
    WF0 --> L31["lane 31 → tid 31"]
```

所以到了这一节，你该先问两个问题，而且它们比「block 设成多大最好」更基础：第一，这群线程有没有正好覆盖该算的输出区域，不多也不少？第二，最后一组如果凑不满，多出来那几个越界的线程，是不是安全退出了、没有去写不该写的内存？如果你的 kernel 需要组内线程互相配合，那么所有线程到达同步点（`__syncthreads()`）的条件也必须一致——不然第一个炸的是**正确性**，根本轮不到谈速度。

> **迁移范围：** `grid`/`block`/`threadIdx` 这些词的用法，在整个 HIP 编程模型里都通用；但 256 这个具体数字，只来自本仓库这一次实验，它**不是** RX 9070 XT 上放之四海皆准的最优 block 大小。block 大小怎么选最合适，要在具体算子里量——第 8 章 Element-Wise 会第一次系统地做这件事。

### 动手验证：线程层级映射表（N=10, block=4）

用一个小规模的例子（N=10, block=4）生成 thread mapping 表，直观展示 grid → workgroup → wavefront → lane 的映射关系。

**预期**：表格显示每个线程的 blockIdx、threadIdx、全局 tid 和所属 wavefront。

In [ ]:
N = 10
block = 4
# `wave_size = 32` 只是 RDNA4/gfx1201 参考环境 的 mapping 示意，不是检测值。
wave_size = 32
# 运行实验后请核对 stdout 的 `ENV ... wave_size=...`；若不是 32，再按运行时值解释分组。
# 如果目标 kernel 报告 wave64，请将上面一行改为 wave_size = 64。

# 生成 thread mapping 表
data = []
for tid in range(N):
    blockIdx = tid // block
    threadIdx = tid % block
    wavefront = threadIdx // wave_size
    lane = threadIdx % wave_size
    data.append({
        "tid": tid,
        "blockIdx.x": blockIdx,
        "threadIdx.x": threadIdx,
        "wavefront": wavefront,
        "lane": lane
    })

print(f"\nThread Mapping Table (N=10, block=4, wave{wave_size}):")
headers = ("tid", "blockIdx.x", "threadIdx.x", "wavefront", "lane")
print(" ".join(f"{header:>12}" for header in headers))
for row in data:
    print(" ".join(f"{row[header]:>12}" for header in headers))

# 边界 lanes 说明
print("\n边界说明:")
print(f"- 总线程数 N={N}")
print(f"- 每个 workgroup 包含 {block} 个线程")
print(f"- 需要 {(N + block - 1) // block} 个 workgroups")
print(f"- 最后一个 workgroup 只有 {N % block if N % block != 0 else block} 个有效线程")
print(f"- 每个 wavefront 最多 {wave_size} 个 lanes（参考示例；请以运行时 ENV 为准）")

## 2.2 workgroup 怎样拆成 wavefront

**一个工作组，会被硬件再切成几个「wavefront」。**

运行时拿到一个 workgroup 后，还会把它拆成一个或多个 wavefront。ROCm 的 `gfx1201` 规格支持两种 wavefront 大小：wave32 和 wave64。我们这台机器在实验里报告 `wave_size=32`，所以 `blockDim.x=256` 这个例子，应该读成**8 个 wave32**，而**不是**「256 个各自独立、同时执行的线程」。

*（图示：一个 256-thread workgroup 在 wave32 下的 8×wave32 分组示意）*

```mermaid
flowchart TB
    WG[一个 256-thread workgroup] --> W0[wave 0: lanes 0-31]
    WG --> W1[wave 1: lanes 32-63]
    WG --> WN[...]
    WG --> W7[wave 7: lanes 224-255]
```

> 仅当此 dispatch 以 wave32 执行时成立。

为什么我们关心的是 wavefront，而不是单个线程？因为硬件执行的最小单位就是 wavefront：同一个 wave 里的所有 lane 共用一段指令，要动一起动。遇到边界或分支时，其中一些 lane 会被暂时关掉，但它们仍然属于同一个 wavefront。

打个比方：一个 workgroup 是一辆大巴上的全体乘客，wavefront 则是其中一排座位——硬件一次只对一排下达同一个动作，整排一起做。本机是 wave32，256 人正好坐满 8 排；要是换成 wave64 的车，每排座位数和分支边界都得重新算。这个比方到这里为止：它帮你建立「成排执行」的直觉，但真实的调度细节，还是要以下文的 LLVM 文档和 profiling 结果为准。

> **迁移范围：** 「wavefront」这个词作为 AMD 的术语，到哪里都适用；但本节「8 个 wave32」的结论，只适用于 256 线程的 block、且 wavefront 大小确实是 32 的情况。换成 wave64 的 kernel，分组和分支边界都要重新算。

## 2.3 wavefront 怎样落到 WGP、CU 和 SIMD

**这些 wavefront，会被放到哪块硬件上去执行？**

先别急着把 wavefront 想象成「钉死」在某张固定的硬件示意图上。我们先固定三个硬件名词：WGP（Workgroup Processor，工作组处理器）、CU（Compute Unit，计算单元）和 SIMD（一条向量指令作用在一组 lane 上的执行单元）。LLVM 的 AMDGPU 文档给出的、可以依赖的保证只有一条：**同一个 workgroup 的所有 wavefront，一定在同一个 WGP 里执行。** 在这个前提下分两种模式：CU 模式下，它们可以落在同一个 CU 里不同的 SIMD 上；WGP 模式下，它们可以分散到同一个 WGP 里不同 CU 的 SIMD 上。WGP 模式是否可用，要根据编译器和目标信息确认，源码或 block 大小本身无法给出答案。

*（图示：WGP/CU/SIMD 的安全读法——LLVM 的执行模式关系，不规定每个 WGP 含几个 CU 或 SIMD）*

```mermaid
flowchart TB
    WG[一个 workgroup 的 wavefronts] --> WGP[同一 WGP: LLVM 的 placement guarantee]
    WGP --> CUmode[CU wavefront mode]
    CUmode --> CUsimd[同一 CU 的不同 SIMD 可执行 waves]
    WGP --> WGPmode[WGP wavefront mode]
    WGPmode --> WGsimd[同一 WGP 内不同 CU 的 SIMD 可执行 waves]
    CUsimd --> Lanes[每条 SIMD 发射向量指令到 active lanes]
    WGsimd --> Lanes
```

RX 9070 XT 的官方规格是 64 个 CU；本章环境里 `rocminfo` 也确实报告 64 个物理 CU 和 `gfx1201`。但有一个容易踩的坑：HIP 里的 `hipDeviceProp_t::multiProcessorCount` 在本机读出来是 **32**，而不是 64。我们在实验输出里就原样记成 `hip_multiprocessor_count=32`，请将它视为运行时诊断字段，不要把它命名为物理 CU 数。换句话说，`multiProcessorCount` 可以帮助描述这次运行环境，物理 CU 总数仍以产品规格和 `rocminfo` 信息为准。

再说 SIMD：它是真正发射向量指令的地方。它解释了为什么「同一个 wave 的 lane 要做同一类工作」很重要——但仅此而已。编程模型本身无法用来推断每条指令要几个周期、每个 WGP 里固定有几个什么单元，或一个 kernel 的并发度有多高。

*（图示：软件概念到硬件单元的对应）*

```mermaid
flowchart LR
    subgraph SW["软件层（你在代码里写的）"]
        WG["workgroup"]
        WF["wavefront"]
        LN["lane"]
    end
    subgraph HW["硬件层（gfx1201）"]
        WGP["WGP"]
        SIMD["SIMD"]
        REG["VGPR / SGPR 寄存器"]
    end
    WG -. 一定落在同一 .-> WGP
    WF -. 被调度到 .-> SIMD
    LN -. 各自占用 .-> REG
```

LLVM 唯一能保证的是：同一 workgroup 的所有 wavefront 落在同一 WGP；至于 wavefront 落到哪条 SIMD、lane 占哪些寄存器，需要结合编译器和当时的调度状态确认，源码本身无法给出答案。

> **迁移范围：** 64 个 CU 是 RX 9070 XT 的产品规格；`multiProcessorCount=32` 是本机 HIP 运行时读出来的值。WGP/CU 两种模式的关系，按 LLVM 文档理解就好，别外推成「WGP 内部一定是某种固定结构」。

## 2.4 分支、EXEC 与有效 lane

**同一个 wavefront 里的线程，如果走了不同的 `if` 分支，会发生什么？**

答案是：整条向量指令流还是照常往前走，但有一个叫**执行掩码（EXEC，Execution Mask）**的东西，负责决定这一刻哪些 lane 的运算是「算数的」。LLVM 的 AMDGPU 文档用嵌套条件描述了它的套路：先存好原来的 `EXEC`，把掩码和当前条件取「与」后执行 then 分支，再把掩码反转去执行 else 分支，最后恢复原样、重新合流。这套机制，正是理解**分支发散（Divergent Control Flow）**的可靠入口——所谓发散，本质就是「哪些 lane 此刻有效」这个集合，随着一段段指令在变化。

*（图示：受控发散时间线——mask 的概念顺序，不主张每段恰好耗费一个周期）*

```mermaid
sequenceDiagram
    participant W as wave32 (示意)
    Note over W: t0: EXEC = lanes 0-31
    W->>W: t1: evaluate predicate
    Note over W: t2: then path, EXEC = lanes selected by predicate
    W->>W: t3: execute then instructions, inactive lanes do no vector work
    Note over W: t4: else path, EXEC = remaining selected lanes
    W->>W: t5: execute else instructions
    Note over W: t6: restore EXEC and reconverge
```

我们可以把 EXEC 想成每排座位上的举手表决：遇到岔路，举手的那部分 lane 走 then 分支，没举手的原地待命——注意，它们**不是**另开一辆车并行跑，只是这一拍不干活；等另一条路走完，两拨再重新合流。失活的 lane 并没有离场，只是被掩码暂时关掉了。这也解释了为什么分支发散会浪费时间：一整排的执行时间，是被最长的那条路拖着走的。

### 动手验证：分支发散受控对照实验

我们用一个只改判断条件的受控实验验证了这一点：让两条路径做完全相同的四条依赖 FP32 乘加，只把 predicate 换成「按 wavefront 一致」或「按 lane 奇偶交替」，后者慢了约 3%。这个数很小、且只属于这套特定的谓词和指令组合——它**不是**「分支发散永远慢 3%」的通用罚单。

核心 kernel 代码：

```cpp
template <BranchMode mode>
__global__ void branch_kernel(const float* input, float* output,
                              std::size_t n) {
    const std::size_t tid =
        static_cast<std::size_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (tid >= n) { return; }

    bool take_a = mode == BranchMode::WaveUniform
        ? (((tid / warpSize) & 1u) == 0u)   // 整个 wave 走同一路
        : ((tid & 1u) == 0u);                // lane 奇偶交替
    float value = input[tid];
    if (take_a) {
        value = fmaf(value, 1.0001f, 0.125f);
        value = fmaf(value, 0.9997f, -0.250f);
        value = fmaf(value, 1.0003f, 0.500f);
        value = fmaf(value, 0.9999f, -0.375f);
    } else {
        value = fmaf(value, 0.9991f, -0.125f);
        value = fmaf(value, 1.0007f, 0.250f);
        value = fmaf(value, 0.9983f, -0.500f);
        value = fmaf(value, 1.0019f, 0.375f);
    }
    output[tid] = value;
}
```

**参考结果**（RX 9070 XT + ROCm 7.13，三进程中位数，`N=16,777,216` FP32）：

| 实现 | 中位数时间（ms） | 吞吐（TFLOPS） | 相对差距 |
| --- | ---: | ---: | ---: |
| `wave-uniform` | 0.237341 | 0.565507 | 基线 |
| `wave-divergent` | 0.245241 | 0.547289 | +3.33% |

下面编译并运行实验，在你的机器上复现：

In [ ]:
chapter2_dir = REPO_ROOT / "code/part0-intro/chapter2"
branch_div_hip = chapter2_dir / "branch_divergence.hip"
branch_div_bin = chapter2_dir / "branch_divergence"

# 编译（使用检测到的架构）
compile_result = run_checked(
    ["hipcc", f"--offload-arch={arch}", "-O3", "-std=c++17",
     str(branch_div_hip), "-o", str(branch_div_bin)],
    timeout=COMPILE_TIMEOUT_S,
    cwd=chapter2_dir,
)
print("编译成功")
if compile_result.stderr.strip():
    print(f"编译器 stderr:\n{compile_result.stderr}")

# 运行实验；编译失败时 run_checked 已停止，不会消费旧 binary。
run_result = run_checked(
    [str(branch_div_bin), "--implementation", "all", "--size", "16777216",
     "--warmup", "10", "--repeat", "50"],
    timeout=FULL_TIMEOUT_S,
    cwd=chapter2_dir,
)
print("\n实验结果:")
print(run_result.stdout)

print("\n解释:")
print("- wave-uniform: 同一 wavefront 内所有 lanes 执行相同分支，无发散")
print("- wave-divergent: 同一 wavefront 内 lanes 执行不同分支，产生发散")
print("- 分支发散会导致性能下降，因为硬件需要串行执行不同分支")

> **迁移范围：** EXEC 这套掩码执行的模型，来自 LLVM AMDGPU 文档，是通用的；上面约 3% 的差异只属于那套受控组合，不作为其他 kernel 的性能预算。在真实算子里，分支发散是否重要、怎样排布数据来缓解，会结合具体场景在 Part 2 反复出现——比如第 9 章 Reduction 里归约树的 lane 参与模式。

## 本章小结

- 一个 kernel 从 launch 出发，先被切成 workgroup，再被切成 wavefront（本机是 wave32）；每个 lane 对应一个全局下标。这是**软件层**的划分。
- 它最终落到 WGP/CU/SIMD 哪块硬件，由 LLVM 的执行模式和编译器决定——唯一能保证的只是「同一 workgroup 的 wavefront 落在同一 WGP」。别从源码或 block 大小反推物理拓扑。
- 同一个 wave 的 lane 走不同分支时，靠 EXEC 掩码轮流执行 then/else 再合流；发散的代价是「整排被最长路径拖住」，而不是两条路并行。
- 这套「软件划分 → 硬件执行」的两层视角是全书的地基。下一章我们补上另一半：片上资源（能同时塞下多少活儿）和内存层级（数据从哪里取）。

## 自我检验

读完本章，你应该能：

1. 能说清 grid、workgroup、wavefront、lane 之间的层级关系，以及 `blockDim.x=256` 为什么在本机是 8 个 wave32。
2. 能区分「软件划分」（workgroup/wavefront）和「硬件落点」（WGP/CU/SIMD），并说出 LLVM 唯一保证的是什么。
3. 能区分 `hipDeviceProp_t::multiProcessorCount=32` 与 `rocminfo` 的 64 physical CU，并说明二者分别代表什么、应如何区分命名。
4. 能解释 EXEC 掩码如何让同一个 wave 的 lane 分别走 then/else 再合流，以及为什么分支发散的代价是「整排被最长路径拖住」。
5. 能把 thread/block/warp/shared memory 这些 CUDA 名词，对应到 HIP 与 AMD 执行模型里的说法。

## 延伸阅读

- [AMD Radeon RX 9070 XT 产品规格](https://www.amd.com/en/products/graphics/desktops/radeon/9000-series/amd-radeon-rx-9070xt.html)：产品级 CU、显存与理论带宽。
- [ROCm GPU specifications](https://rocm.docs.amd.com/en/latest/reference/gpu-specs.html)：用 target 表核对 `gfx1201` 的 wave、LDS 和寄存器资源。
- [LLVM AMDGPU Usage Guide](https://llvm.org/docs/AMDGPUUsage.html)：查 WGP/CU execution mode、wavefront 与 EXEC 的编译器级语义。
- 选做实验：`code/part0-intro/chapter2/`（branch_divergence 受控对照）。